# Logistic Regression Model of NBA All-Star Predictions: 

## Libraries/Requirements: 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, precision_score,
    recall_score, f1_score, accuracy_score
)
from pathlib import Path

## Hyperparameter Selection: 

In [ ]:
class Config:
    """
        Configuration for logistic regression and hybrid (pointwise + pairwise) model.
        Controls:
            - regularization strength search (C_VALUES)
            - solver and optimization settings
            - temporal data splits
            - pairwise ranking construction
            - blending between pointwise and pairwise predictions
        Notes:
            - C_VALUES spans multiple orders of magnitude to test regularization sensitivity
            - TEMP applies post-hoc calibration to probabilities
            - MAX_PAIRS limits pairwise dataset size for efficiency
            - ALPHA controls interpolation between pointwise and pairwise models
    """
    SEED = 47
    C_VALUES = np.logspace(-3, 2, 10)
    TRAIN_END = 2015
    VAL_END = 2021
    PENALTY = 'l2'
    SOLVER = 'lbfgs'
    MAX_ITER = 2000
    TOL = 1e-5
    TEMP = 0.5
    MAX_PAIRS = 200   
    ALPHA = 1.0

def get_project_root():
    """
        Dynamically locates the project root directory.
        Searches upward from the current working directory until a folder
        containing 'source/' is found. This avoids hardcoded paths and
        ensures portability across environments.
        Raises:
            FileNotFoundError if the expected project structure is not found.
    """
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "source").exists(): 
            return parent
    raise FileNotFoundError("Could not find project root (missing 'source' folder)")

ROOT = get_project_root()
DATA_PATH = ROOT / "source" / "cleaned" / "cleaned_data.csv"

Unlike the neural network, this model relies on convex optimization and explicit feature design rather than learned representations. The configuration reflects this by prioritizing stability, reproducibility, and controlled regularization. The logistic regression setup emphasizes stability and interpretability. Regularization is tuned over a wide range, while optimization is configured to reliably converge given the structured feature space.

| Component        | Parameter        | Value | Meaning |
|------------------|------------------|-------|--------|
| Reproducibility  | Seed             | 47 | Fixes randomness for consistent results |
| Model Selection  | C Values         | logspace(-3, 2) | Grid over regularization strengths |
| Regularization   | Penalty          | L2 | Prevents overfitting via weight shrinkage |
| Optimization     | Solver           | lbfgs | Stable optimizer for convex problems |
| Optimization     | Max Iterations   | 2000 | Ensures convergence |
| Optimization     | Tolerance        | 1e-5 | Convergence threshold |
| Calibration      | Temperature      | 0.5 | Controls sharpness of predicted probabilities |
| Ranking          | Max Pairs        | 200 | Limits pairwise comparisons per group |
| Hybrid           | Alpha            | 1.0 | Weight between pointwise and pairwise models |
| Data Split       | Train End        | 2015 | Training cutoff year |
| Data Split       | Val End          | 2021 | Validation cutoff year |
| Data             | Path             | relative to root | Ensures portability across environments |
| Infrastructure   | Project Root     | auto-detected | Avoids hardcoded paths |

### Pairwise Comparison Constructor: 

In [ ]:
def build_pairwise_dataset(X, y, df, max_pairs_per_group=200):
    """
        Constructs a pairwise ranking dataset from pointwise features.
        For each (season, conference) group, all positive (All-Star) vs
        negative (non-All-Star) pairs are formed. Each pair is converted
        into a difference vector:
            x_i - x_j  → label 1  (i should rank above j)
            x_j - x_i  → label 0  (reverse ordering)
        This transforms the problem from classification into a ranking task,
        allowing the model to learn relative comparisons between players
        within the same selection pool.
        Key details:
            - Pairing is restricted within (season, conference) to match the
            actual All-Star selection structure.
            - Symmetric pairs are included to stabilize training.
            - max_pairs_per_group subsamples pairs to control quadratic growth.
        Args:
            X (np.ndarray): Feature matrix (already preprocessed)
            y (pd.Series): Binary labels (1 = All-Star, 0 = non-All-Star)
            df (pd.DataFrame): Original dataframe for grouping
            max_pairs_per_group (int): Maximum number of sampled pairs per group
        Returns:
            X_pairs (np.ndarray): Pairwise difference features
            y_pairs (np.ndarray): Pairwise labels (1 = preferred, 0 = not)
        Notes:
            - This is equivalent to training a linear ranking model using
            logistic regression on feature differences.
            - If no valid pairs exist in a group, it is skipped.
    """
    df = df.reset_index(drop=True)
    y_np = y.values

    X_pairs = []
    y_pairs = []

    for (_, _), idx in df.groupby(['Season Ending Year', 'Conference_East']).groups.items():
        idx = list(idx)

        pos_idx = [i for i in idx if y_np[i] == 1]
        neg_idx = [i for i in idx if y_np[i] == 0]

        if len(pos_idx) == 0 or len(neg_idx) == 0:
            continue

        pairs = [(i, j) for i in pos_idx for j in neg_idx]

        if len(pairs) > max_pairs_per_group:
            pairs = [pairs[k] for k in np.random.choice(len(pairs), max_pairs_per_group, replace=False)]

        for i, j in pairs:
            X_pairs.append(X[i] - X[j])
            y_pairs.append(1)

            X_pairs.append(X[j] - X[i])
            y_pairs.append(0)

    return np.array(X_pairs), np.array(y_pairs)

This transformation converts a classification problem into a preference learning problem, where the model learns which player should be ranked higher rather than whether a player is an All-Star in isolation. The pairwise dataset reframes the problem as a ranking task. Instead of predicting absolute outcomes, the model learns relative comparisons between players within the same selection pool.

| Component        | Step                  | Operation | Meaning |
|------------------|----------------------|----------|--------|
| Grouping         | Selection Pools      | (season, conference) | Restricts comparisons to valid All-Star pools |
| Label Split      | Pos / Neg Separation | y == 1 vs y == 0 | Distinguishes All-Stars from non-All-Stars |
| Pair Generation  | Cartesian Pairs      | (pos, neg) | Constructs all valid ranking comparisons |
| Subsampling      | Pair Limit           | max_pairs_per_group | Prevents combinatorial explosion |
| Feature Transform| Difference Vector    | X[i] − X[j] | Encodes relative comparison between players |
| Symmetry         | Reverse Pair         | (j, i) → label 0 | Ensures balanced binary classification |
| Output           | Pair Features        | X_pairs | Training inputs for ranking model |
| Output           | Pair Labels          | y_pairs | Indicates which player should rank higher |

## Data Pipeline:

In [ ]:
class DataPipeline:
    """
        End-to-end data processing pipeline for the logistic regression model.
        This pipeline handles:
        1. Feature engineering (contextual + interaction-based features)
        2. Temporal train/validation/test splitting
        3. Missing value imputation
        4. Within-group normalization (season + conference)
        5. Global feature scaling
        Design principles:
            - All features are constructed relative to a player's competitive context
            (i.e., within the same season and conference).
            - Group-centering removes global scale effects and forces the model to
            learn *relative differences*, which aligns with the selection process.
            - The pipeline is fit only on training data and applied consistently
            to validation/test to prevent leakage.
        Key transformations:
            - Usage metrics:
                Usage = FGA + 0.44 * FTA
            - Winning-adjusted impact:
                Impact_on_winning = PTS × Team Win %
            - Contextual normalization:
                PTS_conf_z, Impact_conf_z (z-scores within group)
                PTS_conf_share (relative to group max)
            - Interaction features:
                Usage_x_Win, Win_x_AST, Win_x_TRB
            - Temporal features:
                AllStar_Last_Year, AllStar_last_2
            - Group-relative ranking:
                Usage_rank_conf
                Methods:
                    load(path):
                        Reads raw CSV and applies feature engineering.
                    split(df):
                        Splits data chronologically into train / validation / test sets.
                    prepare(train_df, val_df, test_df):
                        - Drops non-feature columns
                        - Applies imputation (median)
                        - Applies group-centering (season, conference)
                        - Applies standard scaling
                        - Returns processed arrays + original dataframes
                    Notes:
                    - Group-centering is critical: it ensures the model compares players
                    within the same selection pool rather than across seasons.
                    - Feature names are stored for interpretability (used in coefficient analysis).
    """
    def __init__(self, cfg):
        self.cfg = cfg
        self.scaler = StandardScaler()
        self.imputer = SimpleImputer(strategy='median')

    def load(self, path):
        df = pd.read_csv(path)
        return self._features(df)

    def _features(self, df):
        group_keys = ['Season Ending Year', 'Conference_East']

        df['Usage'] = df['FGA per game'] + 0.44 * df['FTA per game']
        df['Usage_x_Win'] = df['Usage'] * df['Team Win %']
        df['Impact_on_winning'] = df['PTS per game'] * df['Team Win %']

        pts_mean = df.groupby(group_keys)['PTS per game'].transform('mean')
        pts_std  = df.groupby(group_keys)['PTS per game'].transform('std')

        df['PTS_conf_z'] = (df['PTS per game'] - pts_mean) / (pts_std + 1e-8)
        df['PTS_minus_conf_avg'] = df['PTS per game'] - pts_mean

        pts_max = df.groupby(group_keys)['PTS per game'].transform('max')
        df['PTS_conf_share'] = df['PTS per game'] / (pts_max + 1e-8)

        impact_mean = df.groupby(group_keys)['Impact_on_winning'].transform('mean')
        impact_std  = df.groupby(group_keys)['Impact_on_winning'].transform('std')

        df['Impact_conf_z'] = (
            (df['Impact_on_winning'] - impact_mean) / (impact_std + 1e-8)
        )

        df['Usage_rank_conf'] = df.groupby(group_keys)['Usage'].rank(pct=True)

        df['Win_x_AST'] = df['Team Win %'] * df['AST per game']
        df['Win_x_TRB'] = df['Team Win %'] * df['TRB per game']

        df = df.sort_values(['Player', 'Season Ending Year']).reset_index(drop=True)

        df['AllStar_Last_Year'] = df.groupby('Player')['All Star'].shift(1).fillna(0)
        df['AllStar_last_2'] = (
            df.groupby('Player')['All Star']
            .shift(1)
            .rolling(2)
            .sum()
            .fillna(0)
        )

        df['Impact_x_LastYear'] = df['Impact_on_winning'] * df['AllStar_Last_Year']

        df = df.replace([np.inf, -np.inf], np.nan)
        return df

    def split(self, df):
        train_df = df[df['Season Ending Year'] <= self.cfg.TRAIN_END].copy()
        val_df = df[(df['Season Ending Year'] > self.cfg.TRAIN_END) & (df['Season Ending Year'] <= self.cfg.VAL_END)].copy()
        test_df = df[df['Season Ending Year'] > self.cfg.VAL_END].copy()
        return train_df, val_df, test_df

    def _group_center(self, X, df):
        X_df = pd.DataFrame(X, index=df.index)

        group_means = X_df.groupby(
            [df['Season Ending Year'], df['Conference_East']]
        ).transform('mean')

        centered = (X_df - group_means).fillna(0.0)
        return centered.values

    def prepare(self, train_df, val_df, test_df):
        drop_cols = [
            'All Star', 'Player', 'Season Ending Year',
            'Prev All Stars', 'Conference_East',
            'PosGroup_Backcourt', 'PosGroup_Frontcourt'
        ]

        def split_xy(df):
            X = df.drop(columns=drop_cols)
            y = df['All Star']
            return X, y

        X_train_df, y_train = split_xy(train_df)
        X_val_df, y_val = split_xy(val_df)
        X_test_df, y_test = split_xy(test_df)

        self.feature_names = X_train_df.columns.tolist()

        X_train = self.imputer.fit_transform(X_train_df)
        X_val   = self.imputer.transform(X_val_df)
        X_test  = self.imputer.transform(X_test_df)

        X_train = self._group_center(X_train, train_df)
        X_val   = self._group_center(X_val, val_df)
        X_test  = self._group_center(X_test, test_df)

        X_train = self.scaler.fit_transform(X_train)
        X_val   = self.scaler.transform(X_val)
        X_test  = self.scaler.transform(X_test)

        return (
            X_train, y_train, train_df,
            X_val, y_val, val_df,
            X_test, y_test, test_df
        )

The model’s performance is driven primarily by feature design. By centering and normalizing within selection groups, the input space directly reflects how players are compared in practice rather than relying on raw statistics. Feature engineering is designed to encode relative performance rather than absolute statistics. Most transformations normalize player performance within each (season, conference) group, aligning the feature space with the selection process.

| Component        | Step                  | Operation | Meaning |
|------------------|----------------------|----------|--------|
| Feature Creation | Usage                | FGA + 0.44·FTA | Estimates offensive involvement |
| Feature Creation | Usage × Win          | Usage · Win% | Contextualizes volume with team success |
| Feature Creation | Impact               | PTS · Win% | Measures scoring contribution to winning |
| Normalization    | PTS Z-Score          | (PTS − mean) / std | Relative scoring within group |
| Normalization    | PTS Difference       | PTS − group mean | Absolute deviation from peers |
| Normalization    | PTS Share            | PTS / max | Share of team/conference scoring |
| Normalization    | Impact Z-Score       | standardized impact | Relative contribution to winning |
| Ranking Feature  | Usage Rank           | percentile rank | Relative usage within group |
| Interaction      | Win × AST            | Win% · AST | Playmaking on winning teams |
| Interaction      | Win × TRB            | Win% · TRB | Rebounding on winning teams |
| Temporal Feature | Last Year All-Star   | lag(1) | Captures persistence/reputation |
| Temporal Feature | Last 2 Years         | rolling sum | Captures sustained performance |
| Interaction      | Impact × Last Year   | interaction | Reinforces repeat All-Star signal |
| Cleaning         | Inf Handling         | → NaN | Prevents invalid values |
| Missing Data     | Imputation           | median | Robust to outliers |
| Centering        | Group Centering      | X − group mean | Removes global scale, keeps relative signal |
| Scaling          | Standardization      | mean=0, std=1 | Stabilizes optimization |
| Splitting        | Temporal Split       | train/val/test by year | Prevents leakage across seasons |
| Output           | Feature Matrix       | X | Structured numeric inputs |
| Output           | Labels               | y | Binary All-Star indicator |

## Pairwise and Pointwise Model Class Construction:

In [ ]:
class HybridModel:
    """
        Hybrid logistic regression model combining pointwise classification
        and pairwise ranking.
        The model consists of two components:
            - Pointwise model:
                Standard logistic regression predicting P(All-Star | features).
            - Pairwise model:
                Logistic regression trained on feature differences (x_i - x_j),
                learning relative ordering between players within the same group.
        Final prediction is a convex combination:
            p = alpha * p_point + (1 - alpha) * p_pair; followed by temperature scaling for calibration.
        Design rationale:
            - Pointwise captures absolute likelihood of selection.
            - Pairwise captures *relative dominance* within a selection pool.
            - Blending allows balancing calibration (pointwise) and ranking quality (pairwise).
        Args:
            cfg: Configuration object
            C (float): Inverse regularization strength
        Methods:
            fit(X, y, df):
                Fits both pointwise and pairwise models.
            predict_proba(X, df):
                Returns calibrated probabilities after blending.
        Notes:
            - Pairwise logits are computed manually via linear model parameters.
            - Temperature scaling sharpens or smooths predicted probabilities.
            - Setting alpha = 1 reduces the model to pure logistic regression.
    """
    def __init__(self, cfg, C):
        self.cfg = cfg

        self.pointwise = LogisticRegression(
            C=C,
            penalty=cfg.PENALTY,
            solver=cfg.SOLVER,
            max_iter=cfg.MAX_ITER,
            tol=cfg.TOL,
            n_jobs=-1
        )

        self.pairwise = LogisticRegression(
            C=C,
            penalty=cfg.PENALTY,
            solver=cfg.SOLVER,
            max_iter=cfg.MAX_ITER,
            tol=cfg.TOL,
            n_jobs=-1
        )

        self.alpha = cfg.ALPHA

    def fit(self, X, y, df):
        self.pointwise.fit(X, y)

        X_pair, y_pair = build_pairwise_dataset(X, y, df, self.cfg.MAX_PAIRS)
        self.pairwise.fit(X_pair, y_pair)

    def predict_proba(self, X, df):
        p_point = self.pointwise.predict_proba(X)[:, 1]

        logits_pair = X @ self.pairwise.coef_.T + self.pairwise.intercept_
        logits_pair = logits_pair.squeeze()
        p_pair = 1 / (1 + np.exp(-logits_pair))

        probs = self.alpha * p_point + (1 - self.alpha) * p_pair

        probs = np.clip(probs, 1e-8, 1 - 1e-8)
        logits = np.log(probs) - np.log(1 - probs)
        logits = logits / self.cfg.TEMP
        probs = 1 / (1 + np.exp(-logits))

        return probs
        
def structured_selection(df):
    """
        Applies deterministic roster construction based on predicted probabilities.
        For each (season, conference) group:
            - Selects 2 backcourt + 3 frontcourt players as starters
            - Selects top 7 remaining players as reserves
        This enforces the structural constraints of All-Star selection:
            total = 12 players per conference
            positional quotas for starters
        Args:
            df (pd.DataFrame):
                Must contain:
                    - 'prob' (predicted probability)
                    - 'PosGroup_Backcourt'
                    - 'PosGroup_Frontcourt'
                    - grouping columns
        Returns:
            df (pd.DataFrame):
                Same dataframe with added column:
                - 'pred' (binary selection indicator)
        Notes:
            - This converts probabilistic predictions into a discrete roster.
            - Evaluation metrics (Top-K recall, accuracy, etc.) depend on this step.
            - The model itself is unconstrained; structure is enforced post hoc.
    """
    df = df.copy()
    df['pred'] = 0

    for (season, conf), group in df.groupby(['Season Ending Year', 'Conference_East']):
        bc = group[group['PosGroup_Backcourt'] == 1]
        fc = group[group['PosGroup_Frontcourt'] == 1]

        starters = pd.concat([
            bc.sort_values('prob', ascending=False).head(2),
            fc.sort_values('prob', ascending=False).head(3)
        ])

        remaining = group.drop(index=starters.index)
        reserves = remaining.sort_values('prob', ascending=False).head(7)

        selected = pd.concat([starters, reserves])
        df.loc[selected.index, 'pred'] = 1

    return df

While the neural network encodes structure in its architecture, the logistic regression model approximates it through feature engineering, pairwise comparisons, and post-hoc structured selection. The model combines absolute prediction (pointwise) with relative comparison (pairwise), then converts these scores into a valid roster through structured selection. This bridges standard classification with ranking-based decision making.

| Component        | Step                  | Operation | Meaning |
|------------------|----------------------|----------|--------|
| Pointwise Model  | Logistic Regression  | P(y=1 \| x) | Predicts independent All-Star probability |
| Pairwise Model   | Logistic Regression  | P(i > j) | Learns relative ranking between players |
| Pairwise Input   | Feature Difference   | X[i] − X[j] | Encodes comparative performance |
| Training         | Dual Fit             | pointwise + pairwise | Combines classification and ranking signals |
| Scoring          | Pointwise Prob       | predict_proba(X) | Base probability estimate |
| Scoring          | Pairwise Logits      | X · w_pair | Linear ranking score |
| Conversion       | Sigmoid              | σ(logits) | Converts ranking score to probability |
| Fusion           | Weighted Blend       | α·p_point + (1−α)·p_pair | Combines absolute and relative signals |
| Calibration      | Temperature Scaling  | logits / temp | Controls confidence sharpness |
| Stability        | Clipping             | [1e-8, 1−1e-8] | Prevents numerical instability |
| Grouping         | Selection Pools      | (season, conference) | Restricts competition to valid groups |
| Selection        | Starters             | top-2 BC, top-3 FC | Enforces positional constraints |
| Selection        | Reserves             | top-7 remaining | Completes roster |
| Output           | Binary Prediction    | pred ∈ {0,1} | Final All-Star selection |

## Training Loop: 

In [ ]:
class Trainer:
    """
        Handles hyperparameter tuning and final training for the hybrid logistic model.
        The primary objective is to select the regularization strength (C)
        that best balances ranking quality and probabilistic discrimination.
        Tuning strategy:
            - Train a model for each C in the search grid
            - Evaluate on validation set using:
                (1) AUC → measures global ranking quality
                (2) Top-K Recall → measures correctness of final roster selection
            - Select C by prioritizing Top-K recall, with AUC as a tiebreaker
            This reflects the task structure:
            - The goal is not just classification, but selecting the correct
            subset of players under roster constraints.
        Methods:
            tune(X_train, y_train, train_df, X_val, y_val, val_df):
                Performs grid search over C values.
                Returns the best C based on (TopK, AUC).
            train_final(C, X_train, y_train, train_df):
                Trains a final model on full training data using the chosen C.
        Notes:
            - Top-K recall is computed after structured_selection, making it
            directly aligned with the downstream decision process.
            - AUC ensures the model still maintains strong global ranking behavior.
            - No cross-validation is used; tuning is strictly time-based
            (train → validation split).
    """
    def __init__(self, cfg):
        self.cfg = cfg

    def tune(self, X_train, y_train, train_df, X_val, y_val, val_df):
        results = []

        for C in self.cfg.C_VALUES:
            model = HybridModel(self.cfg, C)
            model.fit(X_train, y_train, train_df)

            probs = model.predict_proba(X_val, val_df)
            auc = roc_auc_score(y_val, probs)

            temp_df = val_df.copy()
            temp_df['prob'] = probs
            temp_df = structured_selection(temp_df)

            true_allstars = y_val.sum()
            selected_correct = ((temp_df['pred'] == 1) & (y_val == 1)).sum()
            topk_recall = selected_correct / (true_allstars + 1e-8)

            results.append((C, auc, topk_recall))
            print(f"C={C:.4f} | AUC={auc:.4f} | TopK={topk_recall:.4f}")

        best_C = sorted(results, key=lambda x: (x[2], x[1]), reverse=True)[0][0]
        print("\nBest C:", best_C)
        return best_C

    def train_final(self, C, X_train, y_train, train_df):
        model = HybridModel(self.cfg, C)
        model.fit(X_train, y_train, train_df)
        return model


Hyperparameter tuning is aligned with the downstream task: selecting the correct players, not just ranking them well. This shifts optimization from standard classification metrics toward structured decision quality. Model selection prioritizes structured performance rather than pure classification accuracy. Top-K recall is used as the primary objective, ensuring the model is optimized for actual roster construction.

| Component        | Step                  | Operation | Meaning |
|------------------|----------------------|----------|--------|
| Hyperparameter   | Regularization Grid  | C ∈ logspace(-3, 2) | Explores strength of L2 penalty |
| Training         | Model Fit            | fit(X_train, y_train) | Learns pointwise + pairwise weights |
| Inference        | Validation Scores    | predict_proba(X_val) | Produces calibrated probabilities |
| Ranking Metric   | AUC                  | roc_auc | Measures global ranking quality |
| Structured Eval  | Selection            | top-12 per group | Converts probabilities into roster |
| Selection Metric | Top-K Recall         | correct / true | Measures how many true All-Stars are selected |
| Model Selection  | Objective            | maximize (TopK, AUC) | Prioritizes selection quality over raw ranking |
| Output           | Best C               | argmax metric | Final regularization choice |
| Final Training   | Refit Model          | train on full train set | Uses best hyperparameter |

## Evaluation Functions: 

In [ ]:
def evaluate(model, X_test, y_test, test_df):
    """
        Evaluates the trained model using both probabilistic and structured metrics.
        The evaluation proceeds in two stages:
            1. Generate probabilities for all players
            2. Apply structured_selection to form final rosters
        Metrics reported:
            - Global:
                AUC, Accuracy, Precision, Recall, F1
            - Per (season, conference):
                Same metrics + Top-K Recall + Top-12 Accuracy
            - Per season (overall):
                Aggregated metrics across both conferences
            - Final summary:
                Average per-season metrics
        Key definitions:
            - Top-K Recall:
                Fraction of true All-Stars correctly selected
            - Top-12 Accuracy:
                Fraction of selected players who are true All-Stars
        Args:
            model: Trained HybridModel
            X_test (np.ndarray): Test features
            y_test (pd.Series): Ground truth labels
            test_df (pd.DataFrame): Original dataframe (used for grouping + structure)
        Notes:
            - Structured selection enforces real roster constraints (2 BC, 3 FC starters + 7 reserves).
            - Metrics reflect both ranking quality (AUC) and discrete selection performance.
            - Per-group evaluation highlights variability across seasons and conferences.
    """
    probs = model.predict_proba(X_test, test_df)

    temp_df = test_df.copy()
    temp_df['prob'] = probs
    temp_df = structured_selection(temp_df)

    y_true = y_test.values
    y_pred = temp_df['pred'].values

    print("\n=== FINAL RESULTS ===")
    print("AUC:", roc_auc_score(y_true, probs))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("\n=== Per-Season (Conference Split) ===")

    season_conf_metrics = []

    for (season, conf), group in temp_df.groupby(['Season Ending Year', 'Conference_East']):
        y_g = group['All Star'].values
        p_g = group['pred'].values
        prob_g = group['prob'].values

        if len(np.unique(y_g)) > 1:
            auc = roc_auc_score(y_g, prob_g)
        else:
            auc = np.nan

        acc = accuracy_score(y_g, p_g)
        prec = precision_score(y_g, p_g, zero_division=0)
        rec = recall_score(y_g, p_g, zero_division=0)
        f1 = f1_score(y_g, p_g, zero_division=0)

        true_allstars = y_g.sum()
        selected_correct = ((p_g == 1) & (y_g == 1)).sum()
        topk_recall = selected_correct / (true_allstars + 1e-8)

        total_selected = p_g.sum()
        top12_acc = selected_correct / (total_selected + 1e-8)

        season_conf_metrics.append((auc, acc, prec, rec, f1, topk_recall, top12_acc))

        print(
            f"{season} Conf {conf} | "
            f"AUC {auc:.4f} | "
            f"Acc {acc:.3f} | "
            f"P {prec:.3f} | R {rec:.3f} | F1 {f1:.3f} | "
            f"Top 12 Recall {topk_recall:.3f} | Top 12 Accuracy {top12_acc:.3f}"
        )

    print("\n=== Per-Season (Overall) ===")

    season_metrics = []

    for season, group in temp_df.groupby('Season Ending Year'):
        y_g = group['All Star'].values
        p_g = group['pred'].values
        prob_g = group['prob'].values

        if len(np.unique(y_g)) > 1:
            auc = roc_auc_score(y_g, prob_g)
        else:
            auc = np.nan

        acc = accuracy_score(y_g, p_g)
        prec = precision_score(y_g, p_g, zero_division=0)
        rec = recall_score(y_g, p_g, zero_division=0)
        f1 = f1_score(y_g, p_g, zero_division=0)

        true_allstars = y_g.sum()
        selected_correct = ((p_g == 1) & (y_g == 1)).sum()
        topk_recall = selected_correct / (true_allstars + 1e-8)

        total_selected = p_g.sum()
        top12_acc = selected_correct / (total_selected + 1e-8)

        season_metrics.append((auc, acc, prec, rec, f1, topk_recall, top12_acc))

        print(
            f"{season} | "
            f"AUC {auc:.4f} | "
            f"Acc {acc:.3f} | "
            f"P {prec:.3f} | R {rec:.3f} | F1 {f1:.3f} | "
            f"Top 12 Recall {topk_recall:.3f} | Top 12 Accuracy {top12_acc:.3f}"
        )

    season_metrics = np.array(season_metrics)

    print("\n=== Avg Per-Season Metrics ===")
    print("AUC            :", np.nanmean(season_metrics[:, 0]))
    print("Accuracy       :", season_metrics[:, 1].mean())
    print("Precision      :", season_metrics[:, 2].mean())
    print("Recall         :", season_metrics[:, 3].mean())
    print("F1             :", season_metrics[:, 4].mean())
    print("Top 12 Recall  :", season_metrics[:, 5].mean())
    print("Top 12 Accuracy:", season_metrics[:, 6].mean())

def print_feature_importance(model, feature_names, top_k=10):
    """
    Displays feature importance for both pointwise and pairwise models.
    Importance is derived from logistic regression coefficients:
        - Positive coefficient → increases likelihood of selection
        - Negative coefficient → decreases likelihood
    Two views are presented:
        1. Pointwise model:
            Interprets features as contributing to absolute All-Star probability
        2. Pairwise model:
            Interprets features as contributing to *winning comparisons* against others
    Outputs:
        - Top positive features (strongest contributors)
        - Top negative features (strongest detractors)
    Args:
        model (HybridModel): Trained model
        feature_names (list): Names of input features
        top_k (int): Number of features to display per category
    Notes:
        - Coefficients are directly comparable due to prior standardization.
        - Pairwise coefficients reflect relative importance, not absolute probability.
        - Differences between pointwise and pairwise rankings can reveal
        which features matter for calibration vs. competition.
    """
    print("\n=== FEATURE IMPORTANCE (POINTWISE) ===")

    coefs = model.pointwise.coef_.flatten()

    df = pd.DataFrame({
        "feature": feature_names,
        "coef": coefs,
        "abs_coef": np.abs(coefs)
    }).sort_values("abs_coef", ascending=False)

    print("\nTop Positive Features:")
    print(df.sort_values("coef", ascending=False).head(top_k)[["feature", "coef"]])

    print("\nTop Negative Features:")
    print(df.sort_values("coef", ascending=True).head(top_k)[["feature", "coef"]])

    print("\n=== FEATURE IMPORTANCE (PAIRWISE) ===")

    coefs_pair = model.pairwise.coef_.flatten()

    df_pair = pd.DataFrame({
        "feature": feature_names,
        "coef": coefs_pair,
        "abs_coef": np.abs(coefs_pair)
    }).sort_values("abs_coef", ascending=False)

    print("\nTop Positive (helps beat others):")
    print(df_pair.sort_values("coef", ascending=False).head(top_k)[["feature", "coef"]])

    print("\nTop Negative (hurts ranking):")
    print(df_pair.sort_values("coef", ascending=True).head(top_k)[["feature", "coef"]])

Notably, several features (e.g., raw scoring metrics) appear negatively weighted after group-centering. This reflects that selection is driven more by *relative performance within a conference-season* than absolute values. 

Evaluation is split into two layers:  
(1) probabilistic ranking (AUC), and  
(2) structured roster construction (Top-K metrics).

Feature importance is examined separately for pointwise and pairwise models, allowing us to distinguish between:
- features that increase raw selection probability
- features that help a player outperform others within the same pool.

| Component | Step | Operation | Meaning |
|----------|------|----------|--------|
| Inference | Probability Prediction | predict_proba(X_test) | Produces calibrated likelihood of All-Star selection |
| Structured Decision | Selection | top-12 per (season, conference) | Enforces real roster constraints (2 BC, 3 FC, 7 reserves) |
| Global Metric | AUC | roc_auc_score | Measures overall ranking quality |
| Global Metric | Accuracy | accuracy_score | Measures classification correctness after selection |
| Global Metric | Precision | precision_score | Fraction of selected players who are true All-Stars |
| Global Metric | Recall | recall_score | Fraction of true All-Stars successfully selected |
| Global Metric | F1 | harmonic mean | Balances precision and recall |
| Group Evaluation | Per (Season, Conf) | grouped metrics | Measures consistency across selection pools |
| Selection Metric | Top-K Recall | correct / true | Measures how many All-Stars are captured |
| Selection Metric | Top-12 Accuracy | correct / selected | Measures quality of final roster |
| Aggregation | Per Season | grouped metrics | Evaluates stability across years |
| Summary | Avg Metrics | mean over seasons | Provides overall performance estimate |
| Interpretability | Pointwise Coefs | model.pointwise.coef_ | Measures direct feature impact on probability |
| Interpretability | Pairwise Coefs | model.pairwise.coef_ | Measures feature impact on ranking between players |
| Feature Ranking | Absolute Value | \|coef\| | Identifies strongest contributors |
| Positive Effects | coef > 0 | increases probability/rank | Feature pushes player toward selection |
| Negative Effects | coef < 0 | decreases probability/rank | Feature pushes player away from selection |

## Runtime: 

In [ ]:
"""
End-to-end training and evaluation pipeline for the hybrid logistic regression model.
Workflow:
    1. Initialize configuration and data pipeline
    2. Load and preprocess dataset
    3. Split data into train / validation / test (time-based)
    4. Prepare features (imputation, group-centering, scaling)
    5. Tune regularization strength (C) using validation data
    6. Train final model on full training set
    7. Inspect feature importance (pointwise + pairwise)
    8. Evaluate on held-out test set using structured selection
Key design choices:
    - Time-based splits prevent leakage across seasons
    - Group-centered features ensure comparisons are made within
    (season, conference) selection pools
    - Hyperparameter tuning prioritizes Top-K recall, aligning with
    the roster selection objective
    - Final evaluation reflects real constraints via structured_selection
Outputs:
    - Best regularization parameter (C)
    - Feature importance rankings
    - Global and per-season performance metrics
Notes:
- The pipeline is fully reproducible given a fixed seed and data path
- Setting ALPHA=1 reduces the model to pure logistic regression
"""
cfg = Config()

pipeline = DataPipeline(cfg)
df = pipeline.load(DATA_PATH)

train_df, val_df, test_df = pipeline.split(df)

(X_train, y_train, train_df, X_val, y_val, val_df, 
    X_test, y_test, test_df) = pipeline.prepare(train_df, val_df, test_df)

In [ ]:
trainer = Trainer(cfg)

best_C = trainer.tune(
    X_train, y_train, train_df,
    X_val, y_val, val_df)

model = trainer.train_final(best_C, X_train, y_train, train_df)

C=0.0010 | AUC=0.9852 | TopK=0.7595
C=0.0036 | AUC=0.9856 | TopK=0.7532
C=0.0129 | AUC=0.9861 | TopK=0.7468
C=0.0464 | AUC=0.9862 | TopK=0.7468
C=0.1668 | AUC=0.9858 | TopK=0.7532
C=0.5995 | AUC=0.9854 | TopK=0.7532
C=2.1544 | AUC=0.9852 | TopK=0.7595
C=7.7426 | AUC=0.9851 | TopK=0.7595
C=27.8256 | AUC=0.9850 | TopK=0.7595
C=100.0000 | AUC=0.9849 | TopK=0.7595

Best C: 2.1544346900318843


In [ ]:
print_feature_importance(model, pipeline.feature_names)
evaluate(model, X_test, y_test, test_df)


=== FEATURE IMPORTANCE (POINTWISE) ===

Top Positive Features:
              feature      coef
23         Team Win %  1.187432
37     PTS_conf_share  1.052594
21               eFG%  0.804823
11       AST per game  0.692949
35         PTS_conf_z  0.663467
38      Impact_conf_z  0.635981
34  Impact_on_winning  0.547252
3    Minutes per game  0.506584
8        ORB per game  0.477106
5        2PA per game  0.457243

Top Negative Features:
               feature      coef
33         Usage_x_Win -0.857216
18                 2P% -0.529635
15         PF per game -0.412201
39     Usage_rank_conf -0.306566
41           Win_x_TRB -0.228757
29              Pos_SF -0.215379
17                 FG% -0.201088
16        PTS per game -0.179578
36  PTS_minus_conf_avg -0.179578
40           Win_x_AST -0.159741

=== FEATURE IMPORTANCE (PAIRWISE) ===

Top Positive (helps beat others):
              feature      coef
35         PTS_conf_z  2.148661
23         Team Win %  1.428741
37     PTS_conf_share  1.28

## Results and Discussion: 

### Design Philosophy

This serves as a controlled baseline against which more expressive models can be meaningfully evaluated. The model is intentionally designed as a constrained, interpretable baseline. Rather than maximizing complexity, the focus is on aligning the modeling assumptions with the structure of the problem: selection is relative, grouped, and constrained. Feature engineering encodes domain knowledge (e.g., usage, impact, conference-relative scoring), while the model itself remains linear to isolate what can be captured without learned interactions. Pairwise ranking is introduced not as a replacement, but as a complement to pointwise probability estimation, reflecting the dual nature of the task: identifying strong candidates and ordering them within a fixed selection pool.

| Principle | Implementation | Rationale |
|----------|--------------|----------|
| Relative evaluation | Group-centering by (season, conference) | Selection is inherently comparative, not absolute |
| Simplicity first | Linear logistic regression | Establishes a strong, interpretable baseline |
| Separation of concerns | Pointwise + pairwise models | Distinguishes probability estimation from ranking |
| Structured decision layer | Top-12 selection (2 BC, 3 FC, 7 reserves) | Matches real-world All-Star constraints |
| Minimal feature engineering | Hand-crafted interaction terms only | Avoids overfitting while encoding domain knowledge |
| Stability over tuning | Wide C sweep with similar results | Indicates robustness of feature space |
| Interpretability | Direct coefficient analysis | Enables understanding of selection drivers |
| Calibration control | Temperature scaling | Adjusts confidence without changing ranking |
| Data integrity | Temporal split (train ≤2015, val ≤2021) | Prevents leakage across seasons |
| Reproducibility | Fixed seed + deterministic pipeline | Ensures consistent results |



Performance is highly stable across regularization strengths, indicating that the feature space is already well-conditioned and largely linearly separable.

AUC is strong (~0.987), meaning the model ranks players well. However, structured recall (~0.71) is noticeably lower, suggesting that translating ranking into discrete roster selection remains the limiting factor.

Overall, the model performs reliably but is constrained by its linear structure, particularly in capturing interaction effects between player performance, team context, and positional constraints.

### Training / Model Summary

| Metric                     | Value            |
|----------------------------|------------------|
| Best C                    | 2.1544           |
| Validation AUC (range)    | 0.9850 – 0.9862  |
| Validation Top-12 (range)  | 0.7468 – 0.7595  |

### Per-Season Performance

| Season | AUC    | Accuracy | Precision | Recall | F1   |
|--------|--------|----------|-----------|--------|------|
| 2022   | 0.9829 | 0.968    | 0.792     | 0.704  | 0.745 |
| 2023   | 0.9848 | 0.957    | 0.708     | 0.630  | 0.667 |
| 2024   | 0.9905 | 0.970    | 0.792     | 0.731  | 0.760 |
| 2025   | 0.9911 | 0.976    | 0.833     | 0.769  | 0.800 |

### Final Model Performance

| Category        | Metric        | Value  |
|-----------------|--------------|--------|
| Global          | AUC          | 0.9872 |
| Global          | Accuracy     | 0.9678 |
| Global          | Precision    | 0.7813 |
| Global          | Recall       | 0.7075 |
| Global          | F1 Score     | 0.7426 |
| Per-Season Avg  | AUC          | 0.9873 |
| Per-Season Avg  | Accuracy     | 0.9677 |
| Per-Season Avg  | Precision    | 0.7813 |
| Per-Season Avg  | Recall       | 0.7083 |
| Per-Season Avg  | F1 Score     | 0.7429 |
| Selection       | Top-K Recall | 0.7083 |
| Selection       | Top-12 Acc   | 0.7813 |

### Feature Importance Summary

Raw scoring features (e.g., PTS per game) appearing as negative signals is not contradictory. After group-centering, the model is learning *relative contribution*, meaning excess scoring without efficiency or team success can be penalized.

Selection is therefore driven less by absolute volume and more by contextualized impact.

| Type | Interpretation | Top Signals |
|------|--------------|------------|
| Pointwise (+) | Increases All-Star probability | Team Win %, PTS_conf_share, eFG%, AST |
| Pointwise (-) | Decreases probability | Usage_x_Win, 2P%, PF |
| Pairwise (+) | Helps beat peers | Impact_on_winning, TRB, Team Win %, PTS_conf_z |
| Pairwise (-) | Hurts ranking | Usage_x_Win, 2P%, raw PTS |

## Model Comparison: Logistic Regression vs Neural Network

The logistic regression model performs strongly as a structured, interpretable baseline, achieving high AUC and stable Top-K recall across seasons. Its strength comes from well-engineered, group-centered features that make the problem close to linearly separable, allowing it to rank players effectively with minimal complexity. However, its linear form limits its ability to capture higher-order interactions between player performance, team context, and positional constraints. 

The neural network, by contrast, is explicitly designed to model these interactions. Through embeddings (conference, position, season), attention across players within the same selection pool, and a multi-objective loss (selection, pairwise ranking, calibration), it learns a richer representation of what differentiates All-Stars. This leads to improved recall and F1, particularly in borderline cases where selection depends on subtle trade-offs rather than dominant single features. 

In short, logistic regression captures the *structure of the data*, while the neural network captures the *structure of the decision*. The former is robust and transparent; the latter is more expressive and better aligned with the combinatorial nature of the selection process.

### Global Performance 

| Metric     | Neural Network | Logistic Regression |
|------------|---------------|---------------------|
| AUC        | 0.9827        | 0.9872              |
| Accuracy   | 0.9740        | 0.9678              |
| Precision  | 0.8333        | 0.7813              |
| Recall     | 0.7547        | 0.7075              |
| F1 Score   | 0.7921        | 0.7426              |

### Per-Season Average Performance

| Metric     | Neural Network | Logistic Regression |
|------------|---------------|---------------------|
| AUC        | 0.9857        | 0.9873              |
| Accuracy   | 0.9739        | 0.9677              |
| Precision  | 0.8333        | 0.7813              |
| Recall     | 0.7553        | 0.7083              |
| F1 Score   | 0.7924        | 0.7429              |

### Selection Quality

| Metric         | Neural Network | Logistic Regression |
|----------------|---------------|---------------------|
| Top-12 Recall   | 0.7553        | 0.7083              |
| Top-12 Accuracy| 0.8333        | 0.7813              |

### Per-Season Performance Comparison

| Season | Model                | AUC    | Accuracy | Precision | Recall | F1   |
|--------|----------------------|--------|----------|-----------|--------|------|
| 2022   | Neural Network       | 0.9816 | 0.983    | 0.917     | 0.815  | 0.863 |
| 2022   | Logistic Regression  | 0.9829 | 0.968    | 0.792     | 0.704  | 0.745 |
| 2023   | Neural Network       | 0.9798 | 0.957    | 0.708     | 0.630  | 0.667 |
| 2023   | Logistic Regression  | 0.9848 | 0.957    | 0.708     | 0.630  | 0.667 |
| 2024   | Neural Network       | 0.9911 | 0.975    | 0.833     | 0.769  | 0.800 |
| 2024   | Logistic Regression  | 0.9905 | 0.970    | 0.792     | 0.731  | 0.760 |
| 2025   | Neural Network       | 0.9903 | 0.981    | 0.875     | 0.808  | 0.840 |
| 2025   | Logistic Regression  | 0.9911 | 0.976    | 0.833     | 0.769  | 0.800 |